<a href="https://colab.research.google.com/github/yingzibu/assignment1-basics/blob/main/cs336_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%cd /content

/content


In [2]:
!git clone https://github.com/yingzibu/assignment1-basics.git

Cloning into 'assignment1-basics'...
remote: Enumerating objects: 263, done.
remote: Total 263 (delta 0), reused 0 (delta 0), pack-reused 263 (from 1)
Receiving objects: 100% (263/263), 14.68 MiB | 22.78 MiB/s, done.
Resolving deltas: 100% (112/112), done.


In [3]:
%cd assignment1-basics

/content/assignment1-basics


In [4]:
!pip -q install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.9/17.9 MB 39.8 MB/s eta 0:00:00


In [5]:
!uv sync

Using CPython 3.13.16 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 67 packages in 1ms
⠙ Preparing packages... (0/66)                                                  warning: `build_system.requires = ["uv-build>=0.11.0,<0.12.0"]` does not contain the current uv version 0.13.0
Prepared 66 packages in 8m 26s
Installed 66 packages in 595ms
 + annotated-types==0.7.0
 + certifi==2026.2.25
 + charset-normalizer==3.4.6
 + click==8.3.1
 + cs336-basics==26.0.0 (from file:///content/assignment1-basics)
 + cuda-bindings==13.2.0
 + cuda-pathfinder==1.5.0
 + cuda-toolkit==13.0.2
 + einops==0.8.2
 + einx==0.4.2
 + filelock==3.25.2
 + frozendict==2.4.7
 + fsspec==2026.3.0
 + gitdb==4.0.12
 + gitpython==3.1.46
 + idna==3.11
 + iniconfig==2.3.0
 + jaxtyping==0.3.9
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.6.1
 + numpy==2.4.4
 + nvidia-cublas==13.1.0.3
 + nvidia-cuda-cupti==13.0.85
 + nvidia-cuda-nvrtc==13.0.88
 + nvidia-cuda-runtime==13.0.

In [6]:
!uv run python --version

Python 3.13.16


In [34]:
!git remote -v

origin	https://github.com/yingzibu/assignment1-basics.git (fetch)
origin	https://github.com/yingzibu/assignment1-basics.git (push)


In [8]:
!uv run pytest -q


tests/test_data.py::test_get_batch FAILED
tests/test_model.py::test_linear FAILED
tests/test_model.py::test_embedding FAILED
tests/test_model.py::test_swiglu FAILED
tests/test_model.py::test_scaled_dot_product_attention FAILED
tests/test_model.py::test_4d_scaled_dot_product_attention FAILED
tests/test_model.py::test_multihead_self_attention FAILED
tests/test_model.py::test_multihead_self_attention_with_rope FAILED
tests/test_model.py::test_transformer_lm FAILED
tests/test_model.py::test_transformer_lm_truncated_input FAILED
tests/test_model.py::test_transformer_block FAILED
tests/test_model.py::test_rmsnorm FAILED
tests/test_model.py::test_rope FAILED
tests/test_model.py::test_silu_matches_pytorch FAILED
tests/test_nn_utils.py::test_softmax_matches_pytorch FAILED
tests/test_nn_utils.py::test_cross_entropy FAILED
tests/test_nn_utils.py::test_gradient_clipping FAILED
tests/test_optimizer.py::test_adamw FAILED
tests/test_optimizer.py::test_get_lr_cosine_schedule FAILED
tests/test_seriali

# 2. Byte-Pair Encoding (BPE) Tokenizer

## 2.1 The Unicode Standard

In [9]:
chr(0)

'\x00'

(a) chr(0) returns '\x00'


In [10]:
print(chr(0))

 


In [13]:
repr(chr(0))

"'\\x00'"

(b) `__repr__()` displays `''\\x00''`, while printing it outputs an invisible NUL character, so it appears blank.

In [11]:
'this is a test' + chr(0) + 'string'

'this is a test\x00string'

In [12]:
print('this is a test' + chr(0) + 'string')

this is a test string


(c) `repr()` will show `\x00` at the corresponding position, while `print()` will show no visible symbol there, making the surrounding text appear continuous.

## 2.2 Unicode Encodings


While the Unicode standard defines a mapping from characters to code points (integers), its impractical to train tokenizers directly on Unicode code points, since the vocabulary would be prohibitively large (around 150K items) and sparse (since many characters are quite rare). Instead, we'll use a Unicode encoding, which converts a Unicode character into a sequence of bytes. The Unicode standard itself defines three encodings: UTF-8, UTF-16 and UTF-32, with UTF-8 being the dominant encoding for the Internet (more than 98% of all webpages).

To encode a Unicode string into UTF-8, we can use the `encode()` function in Python. To access the underlying byte values for a Python  bytes object, we can iterate over it (e.g. call `list()`). Finally, we can use the `decode()` function to decode a UTF-8 byte string into a Unicode string.



In [15]:
utf_type = 'utf-8'
for utf_type in ['utf-8', 'utf-16', 'utf-32']:
    print(utf_type)
    test_string = 'hello! こんにちは！'
    utf_encoded = test_string.encode(utf_type)
    print(utf_encoded)
    print(list(utf_encoded))
    print(utf_encoded.decode(utf_type))
    print(len(test_string), len(utf_encoded))
    print()

utf-8
b'hello! \xe3\x81\x93\xe3\x82\x93\xe3\x81\xab\xe3\x81\xa1\xe3\x81\xaf\xef\xbc\x81'
[104, 101, 108, 108, 111, 33, 32, 227, 129, 147, 227, 130, 147, 227, 129, 171, 227, 129, 161, 227, 129, 175, 239, 188, 129]
hello! こんにちは！
13 25

utf-16
b'\xff\xfeh\x00e\x00l\x00l\x00o\x00!\x00 \x00S0\x930k0a0o0\x01\xff'
[255, 254, 104, 0, 101, 0, 108, 0, 108, 0, 111, 0, 33, 0, 32, 0, 83, 48, 147, 48, 107, 48, 97, 48, 111, 48, 1, 255]
hello! こんにちは！
13 28

utf-32
b'\xff\xfe\x00\x00h\x00\x00\x00e\x00\x00\x00l\x00\x00\x00l\x00\x00\x00o\x00\x00\x00!\x00\x00\x00 \x00\x00\x00S0\x00\x00\x930\x00\x00k0\x00\x00a0\x00\x00o0\x00\x00\x01\xff\x00\x00'
[255, 254, 0, 0, 104, 0, 0, 0, 101, 0, 0, 0, 108, 0, 0, 0, 108, 0, 0, 0, 111, 0, 0, 0, 33, 0, 0, 0, 32, 0, 0, 0, 83, 48, 0, 0, 147, 48, 0, 0, 107, 48, 0, 0, 97, 48, 0, 0, 111, 48, 0, 0, 1, 255, 0, 0]
hello! こんにちは！
13 56



(a) UTF-8 is more space-efficient for English-heavy text because ASCII characters use 1 byte in UTF-8, compared with 2 bytes in UTF-16 and 4 bytes in UTF-32.
UTF-8 is also compatible with ASCII and does not require handling byte-order (endianness) issues.

In [16]:
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

In [22]:
decode_utf8_bytes_to_str_wrong('こ'.encode('utf-8'))

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe3 in position 0: unexpected end of data

(b) The function decodes each byte separately, although the character requires multiple bytes to be decoded together.

(c) 0xA1, 0xFB

In [25]:
%pwd
!grep -n -A 35 -B 5 "def run_train_bpe" tests/adapters.py

560-        A BPE tokenizer that uses the provided vocab, merges, and special tokens.
561-    """
562-    raise NotImplementedError
563-
564-
565:def run_train_bpe(
566-    input_path: str | os.PathLike,
567-    vocab_size: int,
568-    special_tokens: list[str],
569-    **kwargs,
570-) -> tuple[dict[int, bytes], list[tuple[bytes, bytes]]]:
571-    """Given the path to an input corpus, run train a BPE tokenizer and
572-    output its vocabulary and merges.
573-
574-    Args:
575-        input_path (str | os.PathLike): Path to BPE tokenizer training data.
576-        vocab_size (int): Total number of items in the tokenizer's vocabulary (including special tokens).
577-        special_tokens (list[str]): A list of string special tokens to be added to the tokenizer vocabulary.
578-            These strings will never be split into multiple tokens, and will always be
579-            kept as a single token. If these special tokens occur in the `input_path`,
580-            they are treated a

In [26]:
!sed -n '1,240p' tests/test_train_bpe.py

import json
import time

from .adapters import run_train_bpe
from .common import FIXTURES_PATH, gpt2_bytes_to_unicode


def test_train_bpe_speed():
    """
    Ensure that BPE training is relatively efficient by measuring training
    time on this small dataset and throwing an error if it takes more than 1.5 seconds.
    This is a pretty generous upper-bound, it takes 0.38 seconds with the
    reference implementation on my laptop. In contrast, the toy implementation
    takes around 3 seconds.
    """
    input_path = FIXTURES_PATH / "corpus.en"
    start_time = time.time()
    _, _ = run_train_bpe(
        input_path=input_path,
        vocab_size=500,
        special_tokens=["<|endoftext|>"],
    )
    end_time = time.time()
    assert end_time - start_time < 1.5


def test_train_bpe():
    input_path = FIXTURES_PATH / "corpus.en"
    vocab, merges = run_train_bpe(
        input_path=input_path,
        vocab_size=500,
        special_tokens=["<|endoftext|>"],
    )

    # Path to t

In [27]:
bytes(0)

b''

In [30]:
bytes([65]), bytes([0]), bytes([255])

(b'A', b'\x00', b'\xff')

In [31]:
from __future__ import annotations

import heapq
import mmap
import os
import re
from collections import Counter, defaultdict

import regex


Pair = tuple[bytes, bytes]

BYTE_TOKENS = tuple(bytes([i]) for i in range(256))

PRETOKEN_PATTERN = regex.compile(
    r"'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"
)


class _PairCandidate:
    """优先选择频率较高的 pair；同频时选择字典序较大的 pair。"""

    __slots__ = ("count", "pair")

    def __init__(self, count: int, pair: Pair):
        self.count = count
        self.pair = pair

    def __lt__(self, other: _PairCandidate) -> bool:
        if self.count != other.count:
            return self.count > other.count
        return self.pair > other.pair


def _count_pretokens(
    input_path: str | os.PathLike,
    special_tokens: list[str],
) -> Counter[bytes]:
    """特殊 token 隔断文本，各段分别进行 GPT-2 预分词。"""

    counts: Counter[bytes] = Counter()
    special_pattern = None

    if special_tokens:
        alternatives = sorted(
            (token.encode("utf-8") for token in special_tokens),
            key=len,
            reverse=True,
        )
        special_pattern = re.compile(
            b"|".join(re.escape(token) for token in alternatives)
        )

    def count_segment(segment: bytes) -> None:
        text = segment.decode("utf-8")
        counts.update(
            match.group().encode("utf-8")
            for match in PRETOKEN_PATTERN.finditer(text)
        )

    with open(input_path, "rb") as stream:
        if os.fstat(stream.fileno()).st_size == 0:
            return counts

        with mmap.mmap(
            stream.fileno(), 0, access=mmap.ACCESS_READ
        ) as data:
            start = 0

            if special_pattern is not None:
                for match in special_pattern.finditer(data):
                    count_segment(data[start:match.start()])
                    start = match.end()

            count_segment(data[start:])

    return counts


def _merge_word(
    word: tuple[bytes, ...],
    pair: Pair,
) -> tuple[bytes, ...]:
    """从左向右合并，避免重叠位置被重复使用。"""

    first, second = pair
    merged_token = first + second
    result: list[bytes] = []
    i = 0

    while i < len(word):
        if (
            i + 1 < len(word)
            and word[i] == first
            and word[i + 1] == second
        ):
            result.append(merged_token)
            i += 2
        else:
            result.append(word[i])
            i += 1

    return tuple(result)


def _train_from_counts(
    counts: Counter[bytes],
    vocab: dict[int, bytes],
    vocab_size: int,
) -> tuple[dict[int, bytes], list[Pair]]:

    words = [
        tuple(BYTE_TOKENS[b] for b in raw)
        for raw in counts
    ]
    frequencies = list(counts.values())

    pair_counts: Counter[Pair] = Counter()

    # pair -> 包含这个 pair 的预分词编号
    pair_words: dict[Pair, set[int]] = defaultdict(set)

    for word_id, word in enumerate(words):
        local_counts = Counter(zip(word, word[1:]))

        for pair, occurrences in local_counts.items():
            pair_counts[pair] += (
                occurrences * frequencies[word_id]
            )
            pair_words[pair].add(word_id)

    heap = [
        _PairCandidate(count, pair)
        for pair, count in pair_counts.items()
    ]
    heapq.heapify(heap)

    merges: list[Pair] = []

    while len(vocab) < vocab_size and heap:
        candidate = heapq.heappop(heap)
        best_pair = candidate.pair

        # 忽略频率已过期的堆记录
        if candidate.count != pair_counts.get(best_pair, 0):
            continue

        merges.append(best_pair)
        vocab[len(vocab)] = best_pair[0] + best_pair[1]

        changed_pairs: set[Pair] = set()

        # 只处理包含 best_pair 的预分词
        for word_id in tuple(pair_words[best_pair]):
            old_word = words[word_id]
            new_word = _merge_word(old_word, best_pair)

            old_counts = Counter(zip(old_word, old_word[1:]))
            new_counts = Counter(zip(new_word, new_word[1:]))
            frequency = frequencies[word_id]

            for pair in old_counts.keys() | new_counts.keys():
                difference = new_counts[pair] - old_counts[pair]

                if difference:
                    pair_counts[pair] += difference * frequency
                    changed_pairs.add(pair)

                if not new_counts[pair]:
                    pair_words[pair].discard(word_id)
                elif not old_counts[pair]:
                    pair_words[pair].add(word_id)

            words[word_id] = new_word

        # 更新发生变化的 pair
        for pair in changed_pairs:
            count = pair_counts[pair]

            if count > 0:
                heapq.heappush(
                    heap, _PairCandidate(count, pair)
                )
            else:
                del pair_counts[pair]
                pair_words.pop(pair, None)

        # 定期清理过期记录，限制堆的大小
        if len(heap) > 4 * max(len(pair_counts), 1):
            heap = [
                _PairCandidate(count, pair)
                for pair, count in pair_counts.items()
            ]
            heapq.heapify(heap)

    return vocab, merges


def train_bpe(
    input_path: str | os.PathLike,
    vocab_size: int,
    special_tokens: list[str],
) -> tuple[dict[int, bytes], list[tuple[bytes, bytes]]]:

    if not isinstance(vocab_size, int) or isinstance(vocab_size, bool):
        raise TypeError("vocab_size must be an integer")

    if any(not isinstance(token, str) for token in special_tokens):
        raise TypeError("special_tokens must contain strings")

    if any(not token for token in special_tokens):
        raise ValueError("special_tokens cannot contain an empty string")

    # 去除重复的特殊 token，保留输入顺序
    special_tokens = list(dict.fromkeys(special_tokens))

    # ID 0–255：全部单字节 token
    vocab = dict(enumerate(BYTE_TOKENS))

    # 接着加入特殊 token
    for token in special_tokens:
        vocab[len(vocab)] = token.encode("utf-8")

    if vocab_size < len(vocab):
        raise ValueError(
            f"vocab_size must be at least {len(vocab)} "
            "to include all bytes and special tokens"
        )

    counts = _count_pretokens(input_path, special_tokens)

    if vocab_size == len(vocab):
        return vocab, []

    return _train_from_counts(counts, vocab, vocab_size)

In [32]:
!uv run pytest tests/test_train_bpe.py

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-9.0.2, pluggy-1.6.0
rootdir: /content/assignment1-basics
configfile: pyproject.toml
plugins: jaxtyping-0.3.9, timeout-2.4.0
collected 3 items                                                              

tests/test_train_bpe.py::test_train_bpe_speed PASSED
tests/test_train_bpe.py::test_train_bpe PASSED
tests/test_train_bpe.py::test_train_bpe_special_tokens PASSED

============================== 3 passed in 3.10s ===============================


In [56]:
%cd /content/assignment1-basics

/content/assignment1-basics


In [57]:
!git status --short
!git remote -v
!git branch --show-current

origin	https://github.com/yingzibu/assignment1-basics.git (fetch)
origin	https://github.com/yingzibu/assignment1-basics.git (push)
main


In [58]:
!git config user.name 'yingzibu'
!git config user.email "yingzibu@umich.edu"


In [52]:
from google.colab import userdata
import os
os.environ['GH_TOKEN'] = userdata.get("TEMP_TOKEN")

In [60]:
!gh auth status
!git pull --rebase origin main
!gh auth setup-git --hostname github.com --force
!git push -u origin main

github.com
  ✓ Logged in to github.com account yingzibu (GH_TOKEN)
  - Active account: true
  - Git operations protocol: https
  - Token: github_pat_**********************************************************************************
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 1 (from 1)
Unpacking objects: 100% (3/3), 1.23 KiB | 1.23 MiB/s, done.
From https://github.com/yingzibu/assignment1-basics
 * branch            main       -> FETCH_HEAD
   a158843..1ffbeab  main       -> origin/main
Successfully rebased and updated refs/heads/main.
Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 2.87 KiB | 2.87 MiB/s, done.
Total 6 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 

In [55]:
!git add cs336_basics/train_bpe.py tests/adapters.py

In [61]:
# !git add cs336_basics/train_bpe.py tests/adapters.py
!git commit -m "implement BPE training"
!git push -u origin main

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
branch 'main' set up to track 'origin/main'.
Everything up-to-date


In [ ]:
!mkdir -p data


In [70]:
cd data

/content/assignment1-basics/data


In [71]:


!wget https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt
!wget https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-valid.txt


--2026-10-14 03:46:19--  https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt
Resolving huggingface.co (huggingface.co)... 3.167.112.45, 3.167.112.25, 3.167.112.38, ...
Connecting to huggingface.co (huggingface.co)|3.167.112.45|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/645e8da96320b0efe40ade7a/02e40cc51c59a4bc6c51bd7bc9acda4316e208745be060558eaf500cd14e9f96?X-Xet-Cas-Uid=public&user_id=public&response-content-type=text%2Fplain%3B+charset%3Dutf-8&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27TinyStoriesV2-GPT4-train.txt%3B+filename%3D%22TinyStoriesV2-GPT4-train.txt%22%3B&xip=rPyvEWCMK-Y&Expires=1791953179&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjQ1ZThkYTk2MzIwYjBlZmU0MGFkZTdhLzAyZTQwY2M1MWM1OWE0YmM2YzUxYmQ3YmM5YWNkYTQzMTZlMjA4NzQ1YmUwNjA1NThlYWY1MDBjZDE0ZTlmOTZcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZ1c2VyX

In [72]:
!wget https://huggingface.co/datasets/stanford-cs336/owt-sample/resolve/main/owt_train.txt.gz
!gunzip owt_train.txt.gz
!wget https://huggingface.co/datasets/stanford-cs336/owt-sample/resolve/main/owt_valid.txt.gz
!gunzip owt_valid.txt.gz



--2026-10-14 03:46:38--  https://huggingface.co/datasets/stanford-cs336/owt-sample/resolve/main/owt_train.txt.gz
Resolving huggingface.co (huggingface.co)... 3.167.112.45, 3.167.112.25, 3.167.112.38, ...
Connecting to huggingface.co (huggingface.co)|3.167.112.45|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/660add462617cc957fd1bc02/58fb91c989ef15907630f573127276d4db6662ab4aff77b929f901146ee9f208?response-content-type=application%2Fgzip&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27owt_train.txt.gz%3B+filename%3D%22owt_train.txt.gz%22%3B&X-Xet-Cas-Uid=public&user_id=public&xip=rPyvEWCMK-Y&Expires=1791953198&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjYwYWRkNDYyNjE3Y2M5NTdmZDFiYzAyLzU4ZmI5MWM5ODllZjE1OTA3NjMwZjU3MzEyNzI3NmQ0ZGI2NjYyYWI0YWZmNzdiOTI5ZjkwMTE0NmVlOWYyMDhcXD9yZXNwb25zZS1jb250ZW50LXR5cGU9KiZyZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomWC1YZXQ

In [77]:
cd assignment1-basics

/content/assignment1-basics


In [78]:
!python -m cs336_basics.train_bpe_tinystories \
  --input data/TinyStoriesV2-GPT4-train.txt \
  --output artifacts/tinystories_profile \
  --profile

         1655113931 function calls (1655113906 primitive calls) in 1382.289 seconds

   Ordered by: cumulative time
   List reduced from 94 to 20 due to restriction <20>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.069    0.069 1382.289 1382.289 train_bpe.py:200(train_bpe)
        1   11.921   11.921 1368.448 1368.448 train_bpe.py:36(_count_pretokens)
  2717700    5.816    0.000 1354.300    0.000 train_bpe.py:55(count_segment)
  3333195    4.130    0.000 1345.588    0.000 __init__.py:676(update)
  3333193  269.058    0.000 1334.963    0.000 {built-in method _collections._count_elements}
539309868  772.432    0.000 1065.906    0.000 train_bpe.py:57(<genexpr>)
536592171  157.638    0.000  157.638    0.000 {method 'encode' of 'str' objects}
536592168  135.836    0.000  135.836    0.000 {method 'group' of '_regex.Match' objects}
        1    5.852    5.852   13.772   13.772 train_bpe.py:107(_train_from_counts)
  3333221    2.685    0.000    6.495 

In [79]:
import pstats

stats = pstats.Stats(
    "artifacts/tinystories_profile/training.prof"
)
stats.strip_dirs().sort_stats("tottime").print_stats(20)

Wed Oct 14 04:15:17 2026    artifacts/tinystories_profile/training.prof

         1655113931 function calls (1655113906 primitive calls) in 1382.289 seconds

   Ordered by: internal time
   List reduced from 94 to 20 due to restriction <20>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
539309868  772.432    0.000 1065.906    0.000 train_bpe.py:57(<genexpr>)
  3333193  269.058    0.000 1334.963    0.000 {built-in method _collections._count_elements}
536592171  157.638    0.000  157.638    0.000 {method 'encode' of 'str' objects}
536592168  135.836    0.000  135.836    0.000 {method 'group' of '_regex.Match' objects}
        1   11.921   11.921 1368.448 1368.448 train_bpe.py:36(_count_pretokens)
        1    5.852    5.852   13.772   13.772 train_bpe.py:107(_train_from_counts)
  2717700    5.816    0.000 1354.300    0.000 train_bpe.py:55(count_segment)
  3333195    4.130    0.000 1345.588    0.000 __init__.py:676(update)
  3333221    2.685    0.000    6.495    

In [80]:
!git add cs336_basics/train_bpe_tinystories.py cs336_basics/train_bpe_expts_owt.py
!git commit -m "implement BPE training tinystories & expts_owt"
!git push -u origin main

[main a9a2c8c] implement BPE training tinystories & expts_owt
 2 files changed, 437 insertions(+)
 create mode 100644 cs336_basics/train_bpe_expts_owt.py
 create mode 100644 cs336_basics/train_bpe_tinystories.py
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 3.54 KiB | 3.54 MiB/s, done.
Total 5 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 1 local object.
To https://github.com/yingzibu/assignment1-basics.git
   66c7647..a9a2c8c  main -> main
branch 'main' set up to track 'origin/main'.


In [81]:
!python -m cs336_basics.train_bpe_expts_owt \
  --input data/owt_train.txt \
  --output artifacts/owt \
  --tinystories-vocab artifacts/tinystories_profile/vocab.pkl

{
  "vocab_size": 32000,
  "longest_token_bytes": 64,
  "longest_token_repr": "b'\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82'",
  "longest_ordinary_token_bytes": 64,
  "longest_ordinary_token_repr": "b'\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82'",
  "mean_ordinary_token_bytes": 6.33688552767274,
  "longest_ordinary_examples": [
    "b'\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x83\\xc3\\x82\\xc3\\x

In [85]:
!git add cs336_basics/tokenizer.py tests/adapters.py
!git commit -m "implement tokenizer"
!git push -u origin main

[main b6d5c17] implement tokenizer
 2 files changed, 264 insertions(+), 1 deletion(-)
 create mode 100644 cs336_basics/tokenizer.py
Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 2.78 KiB | 2.78 MiB/s, done.
Total 6 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To https://github.com/yingzibu/assignment1-basics.git
   a9a2c8c..b6d5c17  main -> main
branch 'main' set up to track 'origin/main'.


In [83]:
from cs336_basics.tokenizer import Tokenizer

tokenizer = Tokenizer.from_files(
    "artifacts/tinystories_profile/vocab.pkl",
    "artifacts/tinystories_profile/merges.pkl",
    special_tokens=["<|endoftext|>"],
)

text = "Hello, 世界!<|endoftext|>"
ids = tokenizer.encode(text)

print(ids)
print(tokenizer.decode(ids))
assert tokenizer.decode(ids) == text

[1183, 44, 32, 228, 184, 150, 231, 149, 140, 33, 256]
Hello, 世界!<|endoftext|>


In [84]:
!uv run pytest tests/test_tokenizer.py

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-9.0.2, pluggy-1.6.0
rootdir: /content/assignment1-basics
configfile: pyproject.toml
plugins: jaxtyping-0.3.9, timeout-2.4.0
collected 25 items                                                             

tests/test_tokenizer.py::test_roundtrip_empty PASSED
tests/test_tokenizer.py::test_empty_matches_tiktoken PASSED
tests/test_tokenizer.py::test_roundtrip_single_character PASSED
tests/test_tokenizer.py::test_single_character_matches_tiktoken PASSED
tests/test_tokenizer.py::test_roundtrip_single_unicode_character PASSED
tests/test_tokenizer.py::test_single_unicode_character_matches_tiktoken PASSED
tests/test_tokenizer.py::test_roundtrip_ascii_string PASSED
tests/test_tokenizer.py::test_ascii_string_matches_tiktoken PASSED
tests/test_tokenizer.py::test_roundtrip_unicode_string PASSED
tests/test_tokenizer.py::test_unicode_string_matches_tiktoken PASSED
tests/test_token

In [93]:
!git add artifacts
!git commit -m "artifacts"
!git push -u origin main

[main 85177f4] artifacts
 11 files changed, 33413 insertions(+)
 create mode 100644 artifacts/tokenizer_experiments/answers.txt
 create mode 100644 artifacts/tokenizer_experiments/owt_on_owt_sample_ids.json
 create mode 100644 artifacts/tokenizer_experiments/owt_train.npy
 create mode 100644 artifacts/tokenizer_experiments/owt_valid.npy
 create mode 100644 artifacts/tokenizer_experiments/report.json
 create mode 100644 artifacts/tokenizer_experiments/samples.json
 create mode 100644 artifacts/tokenizer_experiments/serialization.json
 create mode 100644 artifacts/tokenizer_experiments/tiny_on_owt_sample_ids.json
 create mode 100644 artifacts/tokenizer_experiments/tiny_on_tiny_sample_ids.json
 create mode 100644 artifacts/tokenizer_experiments/tinystories_train.npy
 create mode 100644 artifacts/tokenizer_experiments/tinystories_valid.npy
Enumerating objects: 17, done.
Counting objects: 100% (17/17), done.
Delta compression using up to 2 threads
Compressing objects: 100% (13/13), done.
er

In [89]:
!python -m cs336_basics.tokenizer_experiments

Sampling 10 documents from each corpus; this scans both training files.
(a) On 10 sampled documents from each corpus, the TinyStories and OpenWebText tokenizers achieved 4.064 and 4.301 bytes/token, respectively.

(b) On the same OpenWebText sample, the TinyStories tokenizer produced 18245 tokens versus 12845 for the OpenWebText tokenizer (3.028 versus 4.301 bytes/token).

(c) Encoding throughput was 0.511 MB/s for TinyStories and 0.123 MB/s for OpenWebText. At those rates, 825 GB would take approximately 448.89 and 1861.36 hours, respectively.

(d) uint16 represents IDs from 0 to 65535 using two bytes per token, covering the ID ranges of the 10K and 32K vocabularies.

Saved results to: /content/assignment1-basics/artifacts/tokenizer_experiments


(a) On 10 sampled documents from each corpus, the TinyStories and OpenWebText tokenizers achieved 4.064 and 4.301 bytes/token, respectively.

(b) On the same OpenWebText sample, the TinyStories tokenizer produced 18245 tokens versus 12845 for the OpenWebText tokenizer (3.028 versus 4.301 bytes/token).

(c) Encoding throughput was 0.511 MB/s for TinyStories and 0.123 MB/s for OpenWebText. At those rates, 825 GB would take approximately 448.89 and 1861.36 hours, respectively.

(d) uint16 represents IDs from 0 to 65535 using two bytes per token, covering the ID ranges of the 10K and 32K vocabularies.


In [92]:
!python -m cs336_basics.tokenizer_experiments --serialize

Sampling 10 documents from each corpus; this scans both training files.
(a) On 10 sampled documents from each corpus, the TinyStories and OpenWebText tokenizers achieved 4.064 and 4.301 bytes/token, respectively.

(b) On the same OpenWebText sample, the TinyStories tokenizer produced 18245 tokens versus 12845 for the OpenWebText tokenizer (3.028 versus 4.301 bytes/token).

(c) Encoding throughput was 0.941 MB/s for TinyStories and 0.456 MB/s for OpenWebText. At those rates, 825 GB would take approximately 243.64 and 502.62 hours, respectively.

(d) uint16 represents IDs from 0 to 65535 using two bytes per token, covering the ID ranges of the 10K and 32K vocabularies.

Encoding tinystories_train...
{'input': 'data/TinyStoriesV2-GPT4-train.txt', 'output': 'artifacts/tokenizer_experiments/tinystories_train.npy', 'dtype': 'uint16', 'tokens': 541229347, 'file_bytes': 1082458950, 'encode_and_write_seconds': 3477.3436464310034}
Encoding tinystories_valid...
{'input': 'data/TinyStoriesV2-GPT4-

In [94]:
import numpy as np

ids = np.load(
    "artifacts/tokenizer_experiments/tinystories_train.npy",
    mmap_mode="r",
)

print(ids.shape, ids.dtype)
print(ids[:20])

(541229347,) uint16
[  10  430  439  259  398  401  283  259  390  496  402  551   46  551
  502  266 1425  263 1569  623]


In [95]:
!git add cs336_basics/tokenizer_experiments.py
!git commit -m "tokenizer experiments"
!git push -u origin main

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/
	owt_train.txt.gz
	owt_valid.txt.gz
	tokenizer_experiments.py
	untitled1
	untitled2

nothing added to commit but untracked files present (use "git add" to track)
Enumerating objects: 17, done.
Counting objects: 100% (17/17), done.
Delta compression using up to 2 threads
Compressing objects: 100% (13/13), done.
error: RPC failed; HTTP 500 curl 22 The requested URL returned error: 500
send-pack: unexpected disconnect while reading sideband packet
Writing objects: 100% (15/15), 4.01 GiB | 45.60 MiB/s, done.
Total 15 (delta 2), reused 2 (delta 0), pack-reused 0
fatal: the remote end hung up unexpectedly
Everything up-to-date


## 3.3 Basic building blocks

In [96]:
!git add cs336_basics/linear.py tests/adapters.py
!git commit -m "linear"
!git push -u origin main

[main 71a87f0] linear
 2 files changed, 62 insertions(+), 1 deletion(-)
 create mode 100644 cs336_basics/linear.py
Enumerating objects: 26, done.
Counting objects: 100% (26/26), done.
Delta compression using up to 2 threads
Compressing objects: 100% (19/19), done.
error: RPC failed; HTTP 500 curl 22 The requested URL returned error: 500
send-pack: unexpected disconnect while reading sideband packet
Writing objects: 100% (21/21), 4.01 GiB | 45.24 MiB/s, done.
Total 21 (delta 6), reused 2 (delta 0), pack-reused 0
fatal: the remote end hung up unexpectedly
Everything up-to-date


In [97]:
!uv run pytest -k test_linear

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-9.0.2, pluggy-1.6.0
rootdir: /content/assignment1-basics
configfile: pyproject.toml
plugins: jaxtyping-0.3.9, timeout-2.4.0
collected 48 items / 47 deselected / 1 selected                                

tests/test_model.py::test_linear PASSED

======================= 1 passed, 47 deselected in 0.55s =======================


### github push

In [ ]:
%%bash
set -e

git fetch origin

if ! git merge-base --is-ancestor origin/main HEAD; then
    echo "远端与本地已有分叉，请贴出 git log --oneline --graph --all -15 的输出。"
    exit 1
fi

# 保存原有提交，方便恢复
git branch "backup-before-cleanup-$(date +%Y%m%d-%H%M%S)"

# 将本地未推送的提交展开为工作目录中的修改
git reset --mixed origin/main

cat >> .gitignore <<'EOF'

data/
artifacts/
.venv/
__pycache__/
*.pyc
*.npy
*.npz
*.pkl
*.prof
EOF

# 提交源码和适配器
git add .gitignore cs336_basics/ tests/adapters.py
git diff --cached --stat
git commit -m "Add implementations and ignore local datasets"
git push origin main

git status -sb

In [100]:
!uv run pytest -k test_embedding

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-9.0.2, pluggy-1.6.0
rootdir: /content/assignment1-basics
configfile: pyproject.toml
plugins: jaxtyping-0.3.9, timeout-2.4.0
collected 48 items / 47 deselected / 1 selected                                

tests/test_model.py::test_embedding PASSED

======================= 1 passed, 47 deselected in 0.40s =======================


In [102]:
!uv run pytest -k test_rmsnorm

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-9.0.2, pluggy-1.6.0
rootdir: /content/assignment1-basics
configfile: pyproject.toml
plugins: jaxtyping-0.3.9, timeout-2.4.0
collected 48 items / 47 deselected / 1 selected                                

tests/test_model.py::test_rmsnorm PASSED

======================= 1 passed, 47 deselected in 0.35s =======================


In [103]:
!uv run pytest -k "test_swiglu or test_silu"

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-9.0.2, pluggy-1.6.0
rootdir: /content/assignment1-basics
configfile: pyproject.toml
plugins: jaxtyping-0.3.9, timeout-2.4.0
collected 48 items / 46 deselected / 2 selected                                

tests/test_model.py::test_swiglu PASSED
tests/test_model.py::test_silu_matches_pytorch PASSED

======================= 2 passed, 46 deselected in 0.31s =======================


In [105]:
!uv run pytest -k test_rope

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-9.0.2, pluggy-1.6.0
rootdir: /content/assignment1-basics
configfile: pyproject.toml
plugins: jaxtyping-0.3.9, timeout-2.4.0
collected 48 items / 47 deselected / 1 selected                                

tests/test_model.py::test_rope PASSED

======================= 1 passed, 47 deselected in 0.27s =======================


In [106]:
!uv run pytest -k test_softmax_matches_pytorch

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-9.0.2, pluggy-1.6.0
rootdir: /content/assignment1-basics
configfile: pyproject.toml
plugins: jaxtyping-0.3.9, timeout-2.4.0
collected 48 items / 47 deselected / 1 selected                                

tests/test_nn_utils.py::test_softmax_matches_pytorch PASSED

======================= 1 passed, 47 deselected in 0.24s =======================


In [108]:
!uv run pytest -k scaled_dot_product_attention

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-9.0.2, pluggy-1.6.0
rootdir: /content/assignment1-basics
configfile: pyproject.toml
plugins: jaxtyping-0.3.9, timeout-2.4.0
collected 48 items / 46 deselected / 2 selected                                

tests/test_model.py::test_scaled_dot_product_attention PASSED
tests/test_model.py::test_4d_scaled_dot_product_attention PASSED

======================= 2 passed, 46 deselected in 0.31s =======================


In [110]:
!uv run pytest -k multihead_self_attention

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-9.0.2, pluggy-1.6.0
rootdir: /content/assignment1-basics
configfile: pyproject.toml
plugins: jaxtyping-0.3.9, timeout-2.4.0
collected 48 items / 46 deselected / 2 selected                                

tests/test_model.py::test_multihead_self_attention PASSED
tests/test_model.py::test_multihead_self_attention_with_rope PASSED

======================= 2 passed, 46 deselected in 0.47s =======================


In [112]:
!uv run pytest -k test_transformer_block

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-9.0.2, pluggy-1.6.0
rootdir: /content/assignment1-basics
configfile: pyproject.toml
plugins: jaxtyping-0.3.9, timeout-2.4.0
collected 48 items / 47 deselected / 1 selected                                

tests/test_model.py::test_transformer_block PASSED

======================= 1 passed, 47 deselected in 0.31s =======================


In [113]:
!uv run pytest -k test_transformer_lm

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-9.0.2, pluggy-1.6.0
rootdir: /content/assignment1-basics
configfile: pyproject.toml
plugins: jaxtyping-0.3.9, timeout-2.4.0
collected 48 items / 46 deselected / 2 selected                                

tests/test_model.py::test_transformer_lm PASSED
tests/test_model.py::test_transformer_lm_truncated_input PASSED

======================= 2 passed, 46 deselected in 0.62s =======================


In [114]:
!python -m cs336_basics.transformer_accounting

# 使用前面 SwiGLU 的默认向上取整规则
!python -m cs336_basics.transformer_accounting --rounding ceil

Batch size = 1; FLOPs count dense matrix multiplications using 2mnp.
FFN rounding: nearest

(a)
Parameters: 1,640,452,800
FP32 weights: 6,561,811,200 bytes = 6.561811200 GB = 6.111162901 GiB
Additional per-layer RoPE caches: 12 MiB

(b)
| Matrix multiplication | FLOPs | TFLOPs |
|---|---:|---:|
| Q/K/V projections | 754,974,720,000 | 0.754974720 |
| Attention output projection | 251,658,240,000 | 0.251658240 |
| QK^T, all heads | 161,061,273,600 | 0.161061274 |
| Attention weights @ V, all heads | 161,061,273,600 | 0.161061274 |
| FFN w1 and w3 | 1,348,888,166,400 | 1.348888166 |
| FFN w2 | 674,444,083,200 | 0.674444083 |
| LM head | 164,682,137,600 | 0.164682138 |
Total: 3,516,769,894,400 FLOPs = 3.516769894 TFLOPs

(c)
Attention projections: 28.6238%
Attention scores and weighted values: 9.1596%
FFN: 57.5338%
LM head: 4.6828%

(d)
| Model | d_ff | TFLOPs | Projections | Scores + values | FFN | LM head |
|---|---:|---:|---:|---:|---:|---:|
| Small | 2048 | 0.291648307 | 19.8808% | 13.

In [116]:
!uv run pytest -k test_cross_entropy

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-9.0.2, pluggy-1.6.0
rootdir: /content/assignment1-basics
configfile: pyproject.toml
plugins: jaxtyping-0.3.9, timeout-2.4.0
collected 48 items / 47 deselected / 1 selected                                

tests/test_nn_utils.py::test_cross_entropy PASSED

======================= 1 passed, 47 deselected in 0.29s =======================


In [117]:
!python -m cs336_basics.learning_rate_tuning

Loss before each update (10 updates total):
| t | lr=1 | lr=10 | lr=100 | lr=1000 |
|---|---:|---:|---:|---:|
| 0 | 2.627140e+01 | 2.627140e+01 | 2.627140e+01 | 2.627140e+01 |
| 1 | 2.523105e+01 | 1.681370e+01 | 2.627140e+01 | 9.483977e+03 |
| 2 | 2.452246e+01 | 1.239434e+01 | 4.507460e+00 | 1.638032e+06 |
| 3 | 2.395941e+01 | 9.697248e+00 | 1.078736e-01 | 1.822136e+08 |
| 4 | 2.348262e+01 | 7.854772e+00 | 1.152566e-16 | 1.475930e+10 |
| 5 | 2.306442e+01 | 6.512506e+00 | 1.284606e-18 | 9.314807e+11 |
| 6 | 2.268932e+01 | 5.492434e+00 | 4.325721e-20 | 4.781918e+13 |
| 7 | 2.234759e+01 | 4.693441e+00 | 2.576862e-21 | 2.057386e+15 |
| 8 | 2.203266e+01 | 4.053156e+00 | 2.210598e-22 | 7.583085e+16 |
| 9 | 2.173987e+01 | 3.530750e+00 | 2.456221e-23 | 2.435013e+18 |

Loss after all 10 updates:
lr=1: loss=2.146575e+01, final/initial=8.170768e-01
lr=10: loss=3.098264e+00, final/initial=1.179330e-01
lr=100: loss=3.318083e-24, final/initial=1.263002e-25
lr=1000: loss=6.903476e+19, final/initial=2

In [118]:
!uv run pytest -k test_adamw

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-9.0.2, pluggy-1.6.0
rootdir: /content/assignment1-basics
configfile: pyproject.toml
plugins: jaxtyping-0.3.9, timeout-2.4.0
collected 48 items / 47 deselected / 1 selected                                

tests/test_optimizer.py::test_adamw PASSED

======================= 1 passed, 47 deselected in 4.52s =======================


In [119]:
!python -m cs336_basics.adamw_accounting

Simplified retained-activation estimate, all floating tensors FP32.
d_ff = 12800/3

(a)
Parameters: 1,635,537,600
Parameter memory: 6,542,150,400 bytes
Gradient memory: 6,542,150,400 bytes
AdamW state memory: 13,084,300,800 bytes
Activation memory: 16,356,614,144 * batch_size bytes

(b)
M(B) = 16.356614144 * B + 26.168601600 GB
Maximum theoretical batch size: 3
Memory at maximum: 75.238444032 GB
Memory at next batch: 91.595058176 GB

(c)
AdamW: 14 * P = 22,897,526,400 FLOPs

(d)
Forward FLOPs per sequence: 3,506,703,564,800
Training FLOPs per effective batch: 10,772,616,248,592,000
Effective compute rate: 247,500,000,000,000 FLOP/s
Training time: 4836.191357 hours
Training time: 201.507973 days


In [120]:
!python -m cs336_basics.adamw_accounting --d-ff 4288 --optimizer-ops 13

Simplified retained-activation estimate, all floating tensors FP32.
d_ff = 4288

(a)
Parameters: 1,640,452,800
Parameter memory: 6,561,811,200 bytes
Gradient memory: 6,561,811,200 bytes
AdamW state memory: 13,123,622,400 bytes
Activation memory: 16,373,391,360 * batch_size bytes

(b)
M(B) = 16.373391360 * B + 26.247244800 GB
Maximum theoretical batch size: 3
Memory at maximum: 75.367418880 GB
Memory at next batch: 91.740810240 GB

(c)
AdamW: 13 * P = 21,325,886,400 FLOPs

(d)
Forward FLOPs per sequence: 3,516,769,894,400
Training FLOPs per effective batch: 10,803,538,441,483,200
Effective compute rate: 247,500,000,000,000 FLOP/s
Training time: 4850.073374 hours
Training time: 202.086391 days


In [122]:
!uv run pytest -k test_get_lr_cosine_schedule

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-9.0.2, pluggy-1.6.0
rootdir: /content/assignment1-basics
configfile: pyproject.toml
plugins: jaxtyping-0.3.9, timeout-2.4.0
collected 48 items / 47 deselected / 1 selected                                

tests/test_optimizer.py::test_get_lr_cosine_schedule PASSED

======================= 1 passed, 47 deselected in 0.18s =======================


In [123]:
!uv run pytest -k test_gradient_clipping

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-9.0.2, pluggy-1.6.0
rootdir: /content/assignment1-basics
configfile: pyproject.toml
plugins: jaxtyping-0.3.9, timeout-2.4.0
collected 48 items / 47 deselected / 1 selected                                

tests/test_nn_utils.py::test_gradient_clipping PASSED

======================= 1 passed, 47 deselected in 0.25s =======================


In [125]:
!uv run pytest -k test_get_batch

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-9.0.2, pluggy-1.6.0
rootdir: /content/assignment1-basics
configfile: pyproject.toml
plugins: jaxtyping-0.3.9, timeout-2.4.0
collected 48 items / 47 deselected / 1 selected                                

tests/test_data.py::test_get_batch PASSED

======================= 1 passed, 47 deselected in 0.53s =======================


In [126]:
!uv run pytest -k test_checkpointing

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-9.0.2, pluggy-1.6.0
rootdir: /content/assignment1-basics
configfile: pyproject.toml
plugins: jaxtyping-0.3.9, timeout-2.4.0
collected 48 items / 47 deselected / 1 selected                                

tests/test_serialization.py::test_checkpointing PASSED

======================= 1 passed, 47 deselected in 1.89s =======================


In [ ]:
# !uv run python -m cs336_basics.train \
#   --resume artifacts/tinystories_training/latest.pt \
#   --max-steps 10000 \
#   --device auto

In [ ]:
!uv run python -m cs336_basics.train \
  --train-data artifacts/tokenizer_experiments/tinystories_train.npy \
  --val-data artifacts/tokenizer_experiments/tinystories_valid.npy \
  --out-dir artifacts/tinystories_training \
  --vocab-size 10000 \
  --batch-size 8 \
  --grad-accum-steps 4 \
  --max-steps 5000 \
  --save-every 100 \
  --device auto

{
  "train": {
    "path": "/content/assignment1-basics/artifacts/tokenizer_experiments/tinystories_train.npy",
    "tokens": 541229347,
    "dtype": "uint16",
    "checked_tokens": 1000000,
    "min_checked_id": 10,
    "max_checked_id": 9997
  },
  "validation": {
    "path": "/content/assignment1-basics/artifacts/tokenizer_experiments/tinystories_valid.npy",
    "tokens": 5465883,
    "dtype": "uint16",
    "checked_tokens": 1000000,
    "min_checked_id": 10,
    "max_checked_id": 9998
  },
  "config": {
    "vocab_size": 10000,
    "context_length": 256,
    "d_model": 256,
    "num_layers": 4,
    "num_heads": 8,
    "d_ff": 704,
    "rope_theta": 10000.0,
    "rms_eps": 1e-05,
    "batch_size": 8,
    "grad_accum_steps": 4,
    "max_steps": 5000,
    "max_lr": 0.0003,
    "min_lr": 3e-05,
    "warmup_iters": 100,
    "cosine_cycle_iters": 5000,
    "beta1": 0.9,
    "beta2": 0.999,
    "adam_eps": 1e-08,
    "weight_decay": 0.1,
    "max_l2_norm": 1.0,
    "log_every": 10,
    "e